# Freddie Mac 2015–2023 Modeling Dataset

This notebook merges each cleaned origination cohort with its eligible 24-month target and validates the resulting one-row-per-loan dataset. The merge preserves the observed event rate; no resampling or class weighting belongs in this data-preparation step.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.freddie_modeling_dataset import (
    LABEL_COLUMNS,
    TARGET_AUDIT_COLUMNS,
    run_modeling_dataset_pipeline,
)

print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

Notebook directory: e:\capstone_project\mortgage-model-reliability\notebooks
Project root: e:\capstone_project\mortgage-model-reliability


## 1. Define the merge contract

The output contains only target-eligible loans. A loan is joined by both `SOURCE YEAR` and `LOAN IDENTIFIER`, and its cohort must agree across the origination and target artifacts.

In [2]:
YEARS = list(range(2015, 2024))
SELECTED_HORIZON = 24
EXPECTED_LOANS_PER_YEAR = 50000
EXPECTED_ELIGIBLE_LOANS = 437668

PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)

input_paths = [
    PROCESSED_DATA_DIRECTORY
    / f"origination_{year}_clean.parquet"
    for year in YEARS
]
input_paths.append(
    PROCESSED_DATA_DIRECTORY
    / "performance_target_2015_2023_24m.parquet"
)

file_check = pd.DataFrame(
    {
        "path": [str(path) for path in input_paths],
        "exists": [path.exists() for path in input_paths],
    }
)
display(file_check)

missing_paths = file_check.loc[
    ~file_check["exists"],
    "path",
].tolist()

if missing_paths:
    raise FileNotFoundError(
        f"Missing modeling-dataset inputs: {missing_paths}"
    )

,path,exists
0,e:\capstone_project\mortgage-model-reliability...,True
1,e:\capstone_project\mortgage-model-reliability...,True
2,e:\capstone_project\mortgage-model-reliability...,True
3,e:\capstone_project\mortgage-model-reliability...,True
4,e:\capstone_project\mortgage-model-reliability...,True
5,e:\capstone_project\mortgage-model-reliability...,True
6,e:\capstone_project\mortgage-model-reliability...,True
7,e:\capstone_project\mortgage-model-reliability...,True
8,e:\capstone_project\mortgage-model-reliability...,True
9,e:\capstone_project\mortgage-model-reliability...,True


## 2. Merge, validate, save, and verify the readback

The pipeline rejects duplicate or unmatched keys, year/cohort disagreement, missing or non-binary labels, an incorrect horizon, event-period inconsistencies, and any row loss relative to the validated target file.

In [3]:
results = run_modeling_dataset_pipeline(
    years=YEARS,
    input_directory=PROCESSED_DATA_DIRECTORY,
    selected_horizon=SELECTED_HORIZON,
    expected_origination_loans_per_year=(
        EXPECTED_LOANS_PER_YEAR
    ),
)

print("Saved outputs:")
for name, path in results["output_paths"].items():
    print(f"{name}: {path}")

Saved outputs:
modeling_dataset: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m.parquet
annual_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m_summary.csv
validation: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m_validation.json


## 3. Reconcile annual row counts and outcomes

In [4]:
annual_summary = results["annual_summary"].copy()
display(
    annual_summary.drop(
        columns="composite_event_rate"
    ).round(3)
)

assert annual_summary["counts_reconcile"].all()
assert annual_summary["merged_loans"].sum() == (
    EXPECTED_ELIGIBLE_LOANS
)
assert results["validation"][
    "readback_validation_matches"
]
assert results["validation"][
    "readback_annual_summary_matches"
]

,SOURCE YEAR,merged_loans,serious_delinquency_events,composite_credit_events,origination_loans,target_eligible_loans,sample_retention_percentage,composite_event_rate_percentage,counts_reconcile
0,2015,47244,248,248,50000,47244,94.488,0.525,True
1,2016,48076,324,324,50000,48076,96.152,0.674,True
2,2017,48278,391,391,50000,48278,96.556,0.81,True
3,2018,47417,1222,1222,50000,47417,94.834,2.577,True
4,2019,48611,2227,2227,50000,48611,97.222,4.581,True
5,2020,49665,802,802,50000,49665,99.33,1.615,True
6,2021,49490,451,451,50000,49490,98.98,0.911,True
7,2022,49386,903,903,50000,49386,98.772,1.828,True
8,2023,49501,876,876,50000,49501,99.002,1.77,True


## 4. Inspect the final dataset

In [5]:
modeling_dataset = results["modeling_dataset"]

print("Shape:", modeling_dataset.shape)
print("Duplicate loan/year keys:", modeling_dataset.duplicated(
    subset=["SOURCE YEAR", "LOAN IDENTIFIER"]
).sum())
print("Missing labels:", modeling_dataset[LABEL_COLUMNS].isna().sum().to_dict())
display(modeling_dataset.head())

Shape: (437668, 59)
Duplicate loan/year keys: 0
Missing labels: {'TARGET_90_PLUS': 0, 'TARGET_COMPOSITE_CREDIT_EVENT': 0}


,SOURCE YEAR,LOAN IDENTIFIER,ORIGINATION YEAR,ORIGINATION QUARTER,ORIGINATION COHORT,CLASSIC FICO,FIRST PAYMENT DATE,FIRST TIME HOMEBUYER INDICATOR,MATURITY DATE,METROPOLITAN STATISTICAL AREA (MSA) OR METROPOLITAN DIVISION,...,POSTAL CODE_CLEAN,NUMBER OF BORROWERS GROUP_CLEAN,HORIZON_MONTHS,TARGET_90_PLUS,TARGET_COMPOSITE_CREDIT_EVENT,FIRST_SERIOUS_DELINQUENCY_PERIOD,FIRST_TERMINAL_CREDIT_EVENT_PERIOD,FIRST_OBSERVED_MONTH,LAST_OBSERVED_MONTH,OBSERVED_MONTHS
0,2015,F15Q10000025,2015,Q1,2015Q1,734,201504,N,203003,NaN,...,812,2+,24,0,0,<NA>,<NA>,-1,66,24
1,2015,F15Q10000031,2015,Q1,2015Q1,797,201504,N,204503,NaN,...,285,2+,24,0,0,<NA>,<NA>,-1,15,16
2,2015,F15Q10000084,2015,Q1,2015Q1,670,201504,N,203003,11460.0,...,481,2+,24,0,0,<NA>,<NA>,-1,131,24
3,2015,F15Q10000087,2015,Q1,2015Q1,731,201503,N,204502,NaN,...,601,2+,24,0,0,<NA>,<NA>,-1,50,24
4,2015,F15Q10000157,2015,Q1,2015Q1,668,201504,N,203003,39340.0,...,846,2+,24,0,0,<NA>,<NA>,-1,131,24


## Modeling boundary

The event-period and observation-window fields are retained only for target auditing. They contain post-origination information and must not enter the predictor matrix. Feature selection and train/evaluation cohort assignment will be handled in the next modeling step.

In [6]:
AUDIT_ONLY_COLUMNS = [
    column
    for column in TARGET_AUDIT_COLUMNS
    if column not in LABEL_COLUMNS
]

print("Labels:", LABEL_COLUMNS)
print("Exclude from predictors:", AUDIT_ONLY_COLUMNS)

Labels: ['TARGET_90_PLUS', 'TARGET_COMPOSITE_CREDIT_EVENT']
Exclude from predictors: ['HORIZON_MONTHS', 'FIRST_SERIOUS_DELINQUENCY_PERIOD', 'FIRST_TERMINAL_CREDIT_EVENT_PERIOD', 'FIRST_OBSERVED_MONTH', 'LAST_OBSERVED_MONTH', 'OBSERVED_MONTHS']


In [7]:
print("Dataset shape:", results["modeling_dataset"].shape)
print()
print("Validation:")
for key, value in results["validation"].items():
    print(f"{key}: {value}")

display(results["annual_summary"].round(3))

Dataset shape: (437668, 59)

Validation:
requested_years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
selected_horizon_months: 24
origination_rows: 450000
target_rows: 437668
modeling_rows: 437668
modeling_columns: 59
duplicate_loan_year_keys: 0
missing_loan_year_keys: 0
missing_labels: 0
observed_years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
observed_horizons: [24]
year_cohort_mismatches: 0
merged_row_count_matches_targets: True
annual_counts_reconcile: True
expected_origination_rows_per_year_match: True
input_target_path: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\performance_target_2015_2023_24m.parquet
output_dataset_path: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\freddie_modeling_dataset_2015_2023_24m.parquet
readback_validation_matches: True
readback_annual_summary_matches: True


,SOURCE YEAR,merged_loans,serious_delinquency_events,composite_credit_events,composite_event_rate,origination_loans,target_eligible_loans,sample_retention_percentage,composite_event_rate_percentage,counts_reconcile
0,2015,47244,248,248,0.005,50000,47244,94.488,0.525,True
1,2016,48076,324,324,0.007,50000,48076,96.152,0.674,True
2,2017,48278,391,391,0.008,50000,48278,96.556,0.81,True
3,2018,47417,1222,1222,0.026,50000,47417,94.834,2.577,True
4,2019,48611,2227,2227,0.046,50000,48611,97.222,4.581,True
5,2020,49665,802,802,0.016,50000,49665,99.33,1.615,True
6,2021,49490,451,451,0.009,50000,49490,98.98,0.911,True
7,2022,49386,903,903,0.018,50000,49386,98.772,1.828,True
8,2023,49501,876,876,0.018,50000,49501,99.002,1.77,True
